# GARCH(1,1) Maximum-Likelihood Optimization

This notebook is the executable narrative for the project. The reusable implementation lives in `src/garch_mle`; the full derivation, assumptions, convergence qualifications and locked results are in `README.md`. GARCH is a new application of the course workflow, not a model claimed to appear in the supplied slides.

## 1. Model, assumptions and constraints

For returns $r_t$,

$$r_t=\mu+\varepsilon_t,\qquad \varepsilon_t=\sqrt{h_t}z_t,\qquad z_t\overset{iid}{\sim}N(0,1),$$

$$h_t=\omega+\alpha\varepsilon_{t-1}^2+\beta h_{t-1}.$$

We impose $\omega>0$, $\alpha,\beta\ge0$, and $\alpha+\beta<1$. The last condition gives finite unconditional variance $\omega/(1-\alpha-\beta)$; it is distinct from the strict-stationarity and finite-fourth-moment conditions.

## 2. Objective, derivatives and numerical method

The Gaussian negative log-likelihood is

$$f(\theta)=\frac12\sum_t\left[\log(2\pi)+\log h_t+\frac{\varepsilon_t^2}{h_t}\right].$$

The variance derivative follows a recursion, so the analytic score is computed in the same forward pass as $h_t$. A log/softmax transform maps an unconstrained vector smoothly into the stationary parameter region. A from-scratch inverse-BFGS method uses Armijo backtracking, an explicit gradient stopping rule, a maximum iteration count and curvature safeguards. Because the likelihood is non-convex, convergence is local; SciPy BFGS provides an independent check.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('src').resolve()))

from garch_mle.diagnostics import ljung_box, standardized_residuals
from garch_mle.estimation import fit_custom_bfgs, fit_scipy_bfgs
from garch_mle.inference import observed_information, profile_persistence
from garch_mle.model import GARCHParameters, filter_variance, simulate_garch11

TRUE_PARAMETERS = GARCHParameters(mu=0.02, omega=0.05, alpha=0.08, beta=0.88)
RETURNS, TRUE_VARIANCE = simulate_garch11(4000, TRUE_PARAMETERS, burn=1000, seed=2026)
TRUE_PARAMETERS

## 3. Controlled synthetic data

The seed, burn-in and true parameters are fixed. Simulation gives a measurable recovery target, but does not validate the GARCH assumptions for real returns.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(RETURNS[:600], linewidth=0.7)
axes[0].set_ylabel('Return (%)')
axes[1].plot(np.sqrt(TRUE_VARIANCE[:600]), color='tab:orange')
axes[1].set_ylabel('True volatility')
axes[1].set_xlabel('Time')
fig.tight_layout();

## 4. Custom BFGS and reference solver

Both solvers receive the same documented starting-value rule, but run independently. Agreement is checked in objective value, parameters and first-order stationarity.

In [ ]:
custom = fit_custom_bfgs(RETURNS)
reference = fit_scipy_bfgs(RETURNS)
comparison = pd.DataFrame([
    {
        'method': fit.method,
        **dict(zip(['mu', 'omega', 'alpha', 'beta'], fit.params.as_array())),
        'persistence': fit.params.persistence,
        'nll': fit.nll,
        'gradient_inf_norm': fit.gradient_inf_norm,
        'iterations': fit.nit,
        'success': fit.success,
    }
    for fit in [custom, reference]
])
comparison

## 5. Observed information and profile likelihood

The Hessian of the negative log-likelihood is obtained by central differences of the analytic score. Wald covariance is transformed back to the natural scale with the delta method. For each fixed persistence $\rho=\alpha+\beta$, all nuisance parameters are re-estimated before applying the one-degree-of-freedom likelihood-ratio cutoff.

In [ ]:
inference = observed_information(custom.unconstrained, RETURNS)
profile = profile_persistence(RETURNS, custom.params, custom.nll)
wald = pd.DataFrame({
    'parameter': ['mu', 'omega', 'alpha', 'beta'],
    'estimate': custom.params.as_array(),
    'standard_error': inference.standard_errors,
    'lower_95': inference.confidence_intervals[:, 0],
    'upper_95': inference.confidence_intervals[:, 1],
})
print(wald.to_string(index=False))
print('Persistence profile 95% CI:', profile.confidence_interval)

## 6. Volatility and residual diagnostics

Residual checks assess model adequacy, not numerical convergence. On real returns, non-normal standardized residuals would motivate robust QMLE inference or a heavy-tailed innovation model.

In [ ]:
fitted_variance = filter_variance(RETURNS, custom.params)
standardized = standardized_residuals(RETURNS, custom.params)
diagnostics = pd.DataFrame([
    {'series': 'standardized residuals', **ljung_box(standardized, 20).__dict__},
    {'series': 'squared standardized residuals', **ljung_box(standardized**2, 20).__dict__},
])
print(diagnostics.to_string(index=False))
plt.figure(figsize=(11, 3.5))
plt.plot(np.sqrt(TRUE_VARIANCE[:600]), label='True volatility')
plt.plot(np.sqrt(fitted_variance[:600]), label='Fitted volatility', alpha=0.8)
plt.legend(); plt.xlabel('Time'); plt.ylabel('Conditional volatility'); plt.tight_layout();